# Linear Regression on AffectNet YOLO emotion data

An end-to-end practical implementation: YOLO annotations → face crops → numerical features → linear regression → MSE/R².

## 1. Dataset and YOLO configuration

This notebook reads the repository's real AffectNet data. Each YOLO label supplies an emotion class and face bounding box.

In [ ]:
from pathlib import Path
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

# This is the repository's actual AffectNet data in YOLO format.
DATASET_ROOT = (Path.cwd().resolve().parent / "dataset" / "YOLO_format")
if not DATASET_ROOT.exists():
    # Also works when the notebook is started from the repository root.
    DATASET_ROOT = Path.cwd().resolve() / "dataset" / "YOLO_format"
YAML_PATH = DATASET_ROOT / "data.yaml"
assert YAML_PATH.exists(), f"Dataset config not found: {YAML_PATH}"

with YAML_PATH.open(encoding="utf-8") as file:
    config = yaml.safe_load(file)
CLASS_NAMES = config["names"]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Dataset: {DATASET_ROOT}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## 2. Shared face-feature pipeline

For each YOLO face crop, we calculate brightness and texture statistics. The same feature columns are used in both practical notebooks.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def yolo_box_to_pixels(xc, yc, bw, bh, width, height):
    """Convert a normalized YOLO box to a safe integer image crop."""
    x1 = max(0, int((xc - bw / 2) * width))
    y1 = max(0, int((yc - bh / 2) * height))
    x2 = min(width, int((xc + bw / 2) * width))
    y2 = min(height, int((yc + bh / 2) * height))
    return x1, y1, x2, y2

def extract_features(face):
    """Numerical facial features used by both models.

    The face is the YOLO-annotated crop.  Features are grayscale brightness and
    texture statistics, including a lower-face region that can reflect smiles.
    """
    if face.ndim == 2:
        gray = face.astype(np.float32)
    else:
        gray = (0.299 * face[..., 0] + 0.587 * face[..., 1] + 0.114 * face[..., 2]).astype(np.float32)
    if gray.max() <= 1.0:
        gray *= 255.0
    lower = gray[gray.shape[0] // 2:, :]
    gx = np.diff(gray, axis=1)
    gy = np.diff(gray, axis=0)
    return [gray.mean(), gray.std(), lower.mean(), lower.std(),
            np.abs(gx).mean(), np.abs(gy).mean()]

def load_yolo_split(split):
    """Read real images + YOLO labels and return one feature row per face box."""
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    rows = []
    skipped = 0
    for image_path in sorted(images_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        label_path = labels_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            skipped += 1
            continue
        try:
            image = mpimg.imread(image_path)
        except Exception:
            skipped += 1
            continue
        height, width = image.shape[:2]
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                continue
            class_id, xc, yc, bw, bh = map(float, values)
            class_id = int(class_id)
            if not 0 <= class_id < NUM_CLASSES:
                continue
            x1, y1, x2, y2 = yolo_box_to_pixels(xc, yc, bw, bh, width, height)
            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                skipped += 1
                continue
            rows.append([*extract_features(crop), class_id, str(image_path)])
    columns = ["mean_intensity", "intensity_std", "lower_mean", "lower_std", "horizontal_texture", "vertical_texture", "class_id", "image_path"]
    frame = pd.DataFrame(rows, columns=columns)
    print(f"{split}: {len(frame):,} annotated faces loaded; {skipped:,} files/crops skipped")
    return frame

train_df = load_yolo_split("train")
valid_df = load_yolo_split("valid")
test_df = load_yolo_split("test")
assert not train_df.empty and not test_df.empty, "Training and test samples are required."
FEATURE_COLUMNS = [column for column in train_df.columns if column not in {"class_id", "image_path"}]
X_train, y_train = train_df[FEATURE_COLUMNS], train_df["class_id"]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df["class_id"]
print("Feature columns:", FEATURE_COLUMNS)
display(train_df.head())
display(pd.DataFrame({"train": y_train.value_counts().sort_index(), "test": y_test.value_counts().sort_index()}, index=range(NUM_CLASSES)).rename(index=dict(enumerate(CLASS_NAMES))).fillna(0).astype(int))

## 3. Train and evaluate

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Emotion IDs are the numeric targets here only to demonstrate Linear Regression.
# The feature pipeline above is shared verbatim with the Decision Tree notebook.
model = LinearRegression()
model.fit(X_train, y_train)
raw_predictions = model.predict(X_test)
predictions = np.clip(np.rint(raw_predictions), 0, NUM_CLASSES - 1).astype(int)

mse = mean_squared_error(y_test, raw_predictions)
r2 = r2_score(y_test, raw_predictions)
rounded_accuracy = accuracy_score(y_test, predictions)
print(f"Test MSE (raw numeric class IDs): {mse:.4f}")
print(f"Test R²  (raw numeric class IDs): {r2:.4f}")
print(f"Rounded-class accuracy (context only): {rounded_accuracy:.4%}")

results = pd.DataFrame({"actual": y_test.map(dict(enumerate(CLASS_NAMES))), "actual_id": y_test,
                        "raw_prediction": raw_predictions, "rounded_prediction": predictions,
                        "predicted": [CLASS_NAMES[i] for i in predictions]})
display(results.head(15))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, raw_predictions, alpha=.25)
plt.plot([0, NUM_CLASSES - 1], [0, NUM_CLASSES - 1], "r--", label="ideal")
plt.xlabel("Actual class ID"); plt.ylabel("Predicted numeric value"); plt.legend(); plt.show()

## Execution record — 13 September 2026

Executed locally against all **17,101 train**, **5,406 validation**, and **2,755 test** YOLO-annotated faces. Test results: **MSE = 5.6317**, **R² = 0.0125**, and rounded-ID accuracy = **14.27%**. The poor fit is expected because numeric emotion IDs do not encode a meaningful continuous target.

## Interpretation and limitation

MSE and R² assess how close the *numeric IDs* are. Emotion labels are nominal: an error from `Anger` (0) to `Contempt` (1) is not intrinsically smaller than an error from `Anger` to `Surprise` (7). Rounding a linear-regression output into a class is therefore only a demonstration, not an appropriate primary emotion classifier. The Decision Tree notebook instead directly learns discrete classes and reports classification metrics.